# 02 — Features & PD model
Narrative around `src/train.py`: the leakage guard, the out-of-time split, calibration,
the three-way benchmark (HGB vs logistic vs grade-alone), and within-grade re-ranking.
Uses the exact pipeline objects the runner uses.

Prereq: `python run_pipeline.py features` has been run.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
np.seterr(all="ignore")
from sklearn.metrics import roc_auc_score

from src import evaluate, features
from src.config import load
from src.db import connect
from src.train import _fit_calibrated, _grade_benchmark

cfg = load()
con = connect(cfg["paths"]["duckdb"], read_only=True)

## Modelling frame + leakage guard

In [2]:
frame = con.execute("SELECT * FROM v_model_frame").fetchdf()
frame = features.engineer(frame)
features.assert_no_leakage(frame.drop(columns=["default_flag"]))          # raises if a forbidden col slipped in
bench = con.execute("SELECT loan_id, lc_grade FROM mart_loan_benchmark").fetchdf()
frame = frame.merge(bench, on="loan_id", how="left")
frame["split_set"].value_counts()

split_set
test     333721
train    306462
Name: count, dtype: int64

## Out-of-time split (by issue_d, never random)

In [3]:
train = frame[frame.split_set == "train"].reset_index(drop=True)
test = frame[frame.split_set == "test"].reset_index(drop=True)
cols = features.ALLOWLIST + features.DERIVED_NUMERIC
ytr, yte = train.default_flag.to_numpy(), test.default_flag.to_numpy()
print(f"train {len(train):,} (DR {ytr.mean():.3f})   test {len(test):,} (DR {yte.mean():.3f})")
print("features:", cols)

train 306,462 (DR 0.132)   test 333,721 (DR 0.149)
features: ['loan_amnt', 'annual_inc', 'dti', 'emp_length_years', 'fico_mid', 'credit_history_months', 'open_acc', 'total_acc', 'revol_bal', 'revol_util', 'delinq_2yrs', 'inq_last_6mths', 'pub_rec', 'mort_acc', 'home_ownership', 'purpose', 'verification_status', 'log_annual_inc', 'loan_to_income']


## Fit the primary model + inspect calibration

In [4]:
model = _fit_calibrated(cfg["model"]["type"], train[cols], ytr, cfg)
p_te = model.predict_proba(test[cols])[:, 1]
evaluate.summary(yte, p_te)

{'auc': 0.6710945959361696,
 'gini': 0.3421891918723392,
 'ks': 0.24837294659399084,
 'brier': 0.12105590345842983,
 'base_rate': 0.14878895844133272,
 'n': 333721}

In [5]:
fig_dir = cfg["paths"]["figures_dir"]
evaluate.plot_calibration(yte, p_te, f"{fig_dir}/nb_calibration.png", "Calibration — out-of-time test")
dec = evaluate.decile_table(yte, p_te)
dec.round(4)

,bucket,n,mean_pd,obs_rate,lift
0,0,33373,0.0356,0.0368,0.2475
1,1,33372,0.0632,0.0662,0.4451
2,2,33372,0.0808,0.0926,0.6223
3,3,33372,0.0986,0.1075,0.7226
4,4,33372,0.1158,0.1297,0.8716
5,5,33372,0.1343,0.1519,1.0209
6,6,33371,0.1536,0.1716,1.1530
7,7,33373,0.1789,0.2007,1.3491
8,8,33372,0.2100,0.2341,1.5731
9,9,33372,0.2647,0.2968,1.9948


## Three-way comparison: primary vs the other model type vs grade-alone

In [6]:
prim = cfg["model"]["type"]
sec = "logistic" if prim == "hgb" else "hgb"
sec_model = _fit_calibrated(sec, train[cols], ytr, cfg)
p_sec = sec_model.predict_proba(test[cols])[:, 1]
grade_bm = _grade_benchmark(train[["lc_grade"]], test[["lc_grade"]], ytr, yte)
pd.Series({
    f"{prim} (primary)": roc_auc_score(yte, p_te),
    f"{sec}": roc_auc_score(yte, p_sec),
    "grade-only": grade_bm["auc"],
}).round(4)

hgb (primary)    0.6711
logistic         0.6584
grade-only       0.6688
dtype: float64

## Does the model re-rank *within* a grade?
Observed default rate by grade x model-PD-decile. If the model adds nothing over grade,
rows are flat.

In [7]:
t = test.copy()
t["pd_hat"] = p_te
t["pd_decile"] = pd.qcut(t.pd_hat.rank(method="first"), 10, labels=False)
(t.groupby(["lc_grade", "pd_decile"]).default_flag.mean().unstack().round(3))

pd_decile,0,1,2,3,4,5,6,7,8,9
lc_grade,,,,,,,,,,
A,0.029,0.047,0.064,0.066,0.073,0.085,0.093,0.108,0.122,0.153
B,0.061,0.076,0.092,0.104,0.115,0.128,0.135,0.149,0.171,0.199
C,0.105,0.116,0.138,0.144,0.164,0.174,0.190,0.210,0.231,0.270
D,0.136,0.163,0.172,0.164,0.197,0.238,0.240,0.265,0.291,0.325
E,0.167,0.145,0.200,0.226,0.250,0.248,0.280,0.324,0.328,0.386
F,0.000,0.000,0.333,0.111,0.404,0.319,0.353,0.415,0.418,0.453
G,NaN,NaN,1.000,NaN,0.167,0.250,0.312,0.556,0.500,0.470


## Calibration by grade — where does PD miss?

In [8]:
(t.groupby("lc_grade")
   .agg(n=("loan_id", "size"), pred_pd=("pd_hat", "mean"), obs_dr=("default_flag", "mean"))
   .round(3))

,n,pred_pd,obs_dr
lc_grade,,,
A,82786,0.076,0.054
B,110329,0.126,0.119
C,90335,0.161,0.195
D,37241,0.188,0.265
E,10994,0.214,0.333
F,1712,0.234,0.423
G,324,0.255,0.460


In [9]:
con.close()